# Future/Past Brain Activity Videos

This notebook creates glass brain videos showing temporal evolution of future and past predictive activity:
1. Load data for comprehension and production
2. Threshold electrodes at 0.1 joint performance
3. Extract lag values from -1000ms to +1000ms (50ms steps)
4. Generate 4 videos:
   - Comprehension Future (green colormap)
   - Comprehension Past (red colormap)
   - Production Future (green colormap)
   - Production Past (red colormap)

In [1]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import numpy as np
import sys
sys.path.append('/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/scripts')
import tfsplt_future_past_utils as pu
from nilearn import plotting
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
res_d = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs"
output_dir = '/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos'
thresh_joint = 0.1

# Lag range for videos: -1000ms to +1000ms in 50ms steps
lag_start = -1000
lag_end = 1000
lag_step = 50
lags_to_plot = np.arange(lag_start, lag_end + lag_step, lag_step)

print(f"Results directory: {res_d}")
print(f"Output directory: {output_dir}")
print(f"Joint threshold: {thresh_joint}")
print(f"Lag range: {lag_start} to {lag_end} ms")
print(f"Number of frames: {len(lags_to_plot)}")

Results directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs
Output directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos
Joint threshold: 0.1
Lag range: -1000 to 1000 ms
Number of frames: 41


## Load Data

In [3]:
# Load comprehension data
print("Loading comprehension data...")
comp_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_comp.csv",
    None
)

# Rename lag columns to numeric values
lag_cols = np.arange(-60000, 60001, 50)
n_lag_cols = len(lag_cols)
new_columns = list(lag_cols) + list(comp_data.columns[n_lag_cols:])
comp_data.columns = new_columns

print(f"Comprehension shape: {comp_data.shape}")


Loading comprehension data...
Comprehension shape: (2644, 2406)


In [4]:
# Load production data
print("Loading production data...")
prod_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_prod.csv",
    None
)

# Rename lag columns to numeric values
new_columns = list(lag_cols) + list(prod_data.columns[n_lag_cols:])
prod_data.columns = new_columns

print(f"Production shape: {prod_data.shape}")


Loading production data...
Production shape: (2644, 2406)


## Threshold Electrodes and Prepare Data

In [5]:
# Get max joint performance for thresholding
def get_max_joint_for_threshold(df, thresh, thresh_lags=None):
    """Get electrodes with max joint performance above threshold."""
    joint_data = df[df['label3'] == 'joint'].copy()
    
    # Get max correlation across all lags for each electrode
    lag_columns = [col for col in joint_data.columns if isinstance(col, (int, float, np.integer, np.floating))]
    joint_data['max_joint'] = joint_data[lag_columns].max(axis=1)
    
    # Filter by threshold
    if thresh_lags is not None:
        joint_filtered = joint_data[joint_data[thresh_lags].max(axis=1) > thresh]
    else:
        joint_filtered = joint_data[joint_data['max_joint'] > thresh]
    
    return joint_filtered[['subject', 'electrode', 'max_joint']]

# Get thresholded electrodes
lags_for_thresh = np.arange(-500, 501, 50).tolist()
comp_thresh = get_max_joint_for_threshold(comp_data, thresh_joint, thresh_lags=lags_for_thresh)
prod_thresh = get_max_joint_for_threshold(prod_data, thresh_joint, thresh_lags=lags_for_thresh)

# remove bad electrodes (visual inspection)
comp_thresh = comp_thresh[~((comp_thresh['subject'] == '798') & 
                            (comp_thresh['electrode'].isin(['G1', 'G65'])))]
prod_thresh = prod_thresh[~((prod_thresh['subject'] == '798') & 
                            (prod_thresh['electrode'] == 'G104'))]


print(f"Comprehension: {len(comp_thresh)} electrodes above threshold")
print(f"Production: {len(prod_thresh)} electrodes above threshold")

Comprehension: 160 electrodes above threshold
Production: 232 electrodes above threshold


In [13]:
# Load electrode coordinates
from tfsplt_brainmap import read_coor

coords_dir = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/data/plotting/brainplot/"
subjects = ["625", "676", "717", "798"]

print("Loading electrode coordinates...")
df_coor = read_coor(coords_dir, subjects)
df_coor.loc[df_coor['subject'] == '717', 'subject'] = '7170'
df_coor = df_coor.rename(columns={"name": "electrode", "MNI_X": "x", "MNI_Y": "y", "MNI_Z": "z"})
df_coor['subject'] = df_coor['subject'].astype(str)

print(f"Loaded coordinates for {len(df_coor)} electrodes")
print(f"Coordinate columns: {df_coor.columns.tolist()}")



print("\nPreparing comprehension data...")
comp_future = pu.prepare_video_data(comp_data, comp_thresh, 'sentence', lags_to_plot, df_coor)
comp_past = pu.prepare_video_data(comp_data, comp_thresh, 'sentence2', lags_to_plot, df_coor)
comp_word = pu.prepare_video_data(comp_data, comp_thresh, 'word', lags_to_plot, df_coor)


print("\nPreparing production data...")
prod_future = pu.prepare_video_data(prod_data, prod_thresh, 'sentence', lags_to_plot, df_coor)
prod_past =   pu.prepare_video_data(prod_data, prod_thresh, 'sentence2', lags_to_plot, df_coor)
prod_word =   pu.prepare_video_data(prod_data, prod_thresh, 'word', lags_to_plot, df_coor)

# comp_future_ratio = pu.prepare_ratio_video_data(comp_data, comp_thresh, 'sentence', 'joint', lags_to_plot, df_coor, joint_max=True)
# comp_past_ratio =   pu.prepare_ratio_video_data(comp_data, comp_thresh, 'sentence2', 'joint', lags_to_plot, df_coor, joint_max=True)
# comp_word_ratio =   pu.prepare_ratio_video_data(comp_data, comp_thresh, 'word', 'joint', lags_to_plot, df_coor, joint_max=True)
# prod_future_ratio = pu.prepare_ratio_video_data(prod_data, prod_thresh, 'sentence', 'joint', lags_to_plot, df_coor, joint_max=True)
# prod_past_ratio =   pu.prepare_ratio_video_data(prod_data, prod_thresh, 'sentence2', 'joint', lags_to_plot, df_coor, joint_max=True)
# prod_word_ratio =   pu.prepare_ratio_video_data(prod_data, prod_thresh, 'word', 'joint', lags_to_plot, df_coor, joint_max=True)

comp_future_ratio, comp_word_ratio, comp_past_ratio = pu.prepare_all_ratio_video_data(comp_data, comp_thresh, lags_to_plot, df_coor, joint_max=True)
prod_future_ratio, prod_word_ratio, prod_past_ratio = pu.prepare_all_ratio_video_data(prod_data, prod_thresh, lags_to_plot, df_coor, joint_max=True)

Loading electrode coordinates...
Loaded coordinates for 666 electrodes
Coordinate columns: ['electrode', 'name_NYUcoor', 'brain_data', 'coordinates', 'T1_X', 'T1_Y', 'T1_Z', 'x', 'y', 'z', 'type', 'reg1', 'reg1_percent', 'reg2', 'reg2_percent', 'reg3', 'reg3_percent', 'reg4', 'reg4_percent', 'subject']

Preparing comprehension data...
  sentence: 158 electrodes, 41 time points
  sentence2: 158 electrodes, 41 time points
  word: 158 electrodes, 41 time points

Preparing production data...
  sentence: 228 electrodes, 41 time points
  sentence2: 228 electrodes, 41 time points
  word: 228 electrodes, 41 time points
Preparing all ratio video data...
  future: 158 electrodes, 41 lags
  word: 158 electrodes, 41 lags
  past: 158 electrodes, 41 lags
Preparing all ratio video data...
  future: 228 electrodes, 41 lags
  word: 228 electrodes, 41 lags
  past: 228 electrodes, 41 lags


In [14]:
# Create custom colormaps
# Future: white -> green (for negative to positive correlations)
future_colors = ['white', 'lightgreen', 'green', 'darkgreen']
future_cmap = LinearSegmentedColormap.from_list('future', future_colors, N=256)

# Past: white -> red (for negative to positive correlations)
past_colors = ['white', 'lightcoral', 'red', 'darkred']
past_cmap = LinearSegmentedColormap.from_list('past', past_colors, N=256)

# Word: white -> orange
word_colors = ['white',  'orange', 'darkorange']
word_cmap = LinearSegmentedColormap.from_list('word', word_colors, N=256)

# Create output directory if it doesn't exist
import os
os.makedirs(output_dir, exist_ok=True)

# Get lag columns (excluding metadata)
lag_columns = [col for col in comp_future.columns if isinstance(col, (int, float, np.integer, np.floating))]

print(f"Will create videos with {len(lag_columns)} frames")
print(f"Lag range: {lag_columns[0]} to {lag_columns[-1]} ms")

Will create videos with 41 frames
Lag range: -1000 to 1000 ms


## Validation - are any ratio sums > 1

In [ ]:
# # Validation - make sure sums<1
# # sum comp_future_ratio, comp_past_ratio, comp_word_ratio for each lag
# comp_ratio_sums = comp_future_ratio[lag_columns].values + comp_past_ratio[lag_columns].values + comp_word_ratio[lag_columns].values
# prod_ratio_sums = prod_future_ratio[lag_columns].values + prod_past_ratio[lag_columns].values + prod_word_ratio[lag_columns].values

In [ ]:
# # Detailed validation checks
# print("=" * 80)
# print("VALIDATION: Checking ratio calculations")
# print("=" * 80)

# # Check 1: Summary statistics of ratio sums
# print("\n1. Ratio Sum Statistics:")
# print(f"Comprehension ratio sums - Min: {comp_ratio_sums.min():.4f}, Max: {comp_ratio_sums.max():.4f}, Mean: {comp_ratio_sums.mean():.4f}")
# print(f"Production ratio sums - Min: {prod_ratio_sums.min():.4f}, Max: {prod_ratio_sums.max():.4f}, Mean: {prod_ratio_sums.mean():.4f}")

# # Check 2: Count how many electrodes/lags exceed 1.0
# comp_violations = (comp_ratio_sums > 1.0).sum()
# prod_violations = (prod_ratio_sums > 1.0).sum()
# print(f"\n2. Ratio sums > 1.0:")
# print(f"Comprehension violations: {comp_violations} / {comp_ratio_sums.size} ({100*comp_violations/comp_ratio_sums.size:.2f}%)")
# print(f"Production violations: {prod_violations} / {prod_ratio_sums.size} ({100*prod_violations/prod_ratio_sums.size:.2f}%)")

# # Check 3: For a specific lag, compare individual components vs joint
# test_lag = 0  # Test at lag 0
# print(f"\n3. Component analysis at lag {test_lag}ms:")

# # Get joint values for comparison (should be the denominator)
# comp_joint = pu.prepare_video_data(comp_data, comp_thresh, 'joint', lags_to_plot, df_coor)
# prod_joint = pu.prepare_video_data(prod_data, prod_thresh, 'joint', lags_to_plot, df_coor)

# print("\nComprehension (first 5 electrodes):")
# for i in range(min(5, len(comp_future))):
#     future_val = comp_future.iloc[i][test_lag]
#     word_val = comp_word.iloc[i][test_lag]
#     past_val = comp_past.iloc[i][test_lag]
#     joint_val = comp_joint.iloc[i][test_lag]
    
#     future_ratio = comp_future_ratio.iloc[i][test_lag]
#     word_ratio = comp_word_ratio.iloc[i][test_lag]
#     past_ratio = comp_past_ratio.iloc[i][test_lag]
    
#     print(f"  Electrode {i}:")
#     print(f"    Raw: future={future_val:.4f}, word={word_val:.4f}, past={past_val:.4f}, joint={joint_val:.4f}")
#     print(f"    Ratios: future={future_ratio:.4f}, word={word_ratio:.4f}, past={past_ratio:.4f}")
#     print(f"    Sum of ratios: {future_ratio + word_ratio + past_ratio:.4f}")
#     print(f"    Max component: {max(future_val, word_val, past_val):.4f} vs joint: {joint_val:.4f}")

# # Check 4: Verify that joint >= max(future, word, past) for all lags
# print("\n4. Checking if joint >= max(components):")
# comp_max_component = np.maximum.reduce([comp_future[lag_columns].values, 
#                                          comp_word[lag_columns].values, 
#                                          comp_past[lag_columns].values])
# comp_joint_values = comp_joint[lag_columns].values
# comp_violations_joint = (comp_max_component > comp_joint_values).sum()

# prod_max_component = np.maximum.reduce([prod_future[lag_columns].values, 
#                                          prod_word[lag_columns].values, 
#                                          prod_past[lag_columns].values])
# prod_joint_values = prod_joint[lag_columns].values
# prod_violations_joint = (prod_max_component > prod_joint_values).sum()

# print(f"Comprehension: {comp_violations_joint} cases where max(component) > joint")
# print(f"Production: {prod_violations_joint} cases where max(component) > joint")

# # Check 5: Look at specific cases where ratio sum > 1
# if comp_violations > 0:
#     print("\n5. Example cases where comprehension ratio sum > 1.0:")
#     # Find indices where sum > 1
#     bad_indices = np.where(comp_ratio_sums > 1.0)
#     # Show first few examples
#     for idx in range(min(3, len(bad_indices[0]))):
#         row_idx = bad_indices[0][idx]
#         col_idx = bad_indices[1][idx]
#         lag_val = lag_columns[col_idx]
        
#         print(f"\n  Electrode {row_idx}, Lag {lag_val}ms:")
#         print(f"    Future: raw={comp_future.iloc[row_idx][lag_val]:.4f}, ratio={comp_future_ratio.iloc[row_idx][lag_val]:.4f}")
#         print(f"    Word: raw={comp_word.iloc[row_idx][lag_val]:.4f}, ratio={comp_word_ratio.iloc[row_idx][lag_val]:.4f}")
#         print(f"    Past: raw={comp_past.iloc[row_idx][lag_val]:.4f}, ratio={comp_past_ratio.iloc[row_idx][lag_val]:.4f}")
#         print(f"    Joint: {comp_joint.iloc[row_idx][lag_val]:.4f}")
#         print(f"    Sum of ratios: {comp_ratio_sums[row_idx, col_idx]:.4f}")

# print("\n" + "=" * 80)

VALIDATION: Checking ratio calculations

1. Ratio Sum Statistics:
Comprehension ratio sums - Min: 1.0000, Max: 1.0000, Mean: 1.0000
Production ratio sums - Min: 1.0000, Max: 1.0000, Mean: 1.0000

2. Ratio sums > 1.0:
Comprehension violations: 3076 / 6478 (47.48%)
Production violations: 4618 / 9348 (49.40%)

3. Component analysis at lag 0ms:
  joint: 158 electrodes, 41 time points
  joint: 228 electrodes, 41 time points

Comprehension (first 5 electrodes):
  Electrode 0:
    Raw: future=0.0457, word=0.0063, past=0.0335, joint=0.1584
    Ratios: future=0.2238, word=0.0355, past=0.7407
    Sum of ratios: 1.0000
    Max component: 0.0457 vs joint: 0.1584
  Electrode 1:
    Raw: future=0.0118, word=0.0003, past=0.0449, joint=0.1855
    Ratios: future=0.4751, word=0.0135, past=0.5114
    Sum of ratios: 1.0000
    Max component: 0.0449 vs joint: 0.1855
  Electrode 2:
    Raw: future=0.0355, word=0.0056, past=0.0574, joint=0.1364
    Ratios: future=0.3137, word=0.0485, past=0.6378
    Sum of r

## Interactive Plotly Visualization

Create an interactive scrollable 3D brain visualization with separate rows for future, word, and past activity.

In [15]:
# Prepare comprehension data (word data needs to be loaded)
print("=" * 80)
print("Plotting comprehension timecourses...")
print("=" * 80)

comp_html_fig = pu.create_interactive_brain_viz_html(
    comp_future, 
    comp_word, 
    comp_past, 
    lags_to_plot,
    output_path=f"{output_dir}/comprehension_all_bands_interactive.html",
    title_prefix="Comprehension",
    vmin=0,
    vmax=0.25
)

print("=" * 80)
print("Plotting production timecourses...")
print("=" * 80)
prod_html_fig = pu.create_interactive_brain_viz_html(
    prod_future, 
    prod_word, 
    prod_past, 
    lags_to_plot,
    output_path=f"{output_dir}/production_all_bands_interactive.html",
    title_prefix="Production",
    vmin=0,
    vmax=0.25
)

print("=" * 80)
print("Plotting comprehension ratio timecourses...")
print("=" * 80)
comp_ratio_html_fig = pu.create_interactive_brain_viz_html(
    comp_future_ratio, 
    comp_word_ratio, 
    comp_past_ratio, 
    lags_to_plot,
    output_path=f"{output_dir}/comprehension_all_bands_ratio_interactive.html",
    title_prefix="Comprehension Ratio",
    vmin=0,
    vmax=0.8
)

print("=" * 80)
print("Plotting production ratio timecourses...")
print("=" * 80)
prod_ratio_html_fig = pu.create_interactive_brain_viz_html(
    prod_future_ratio, 
    prod_word_ratio, 
    prod_past_ratio, 
    lags_to_plot,
    output_path=f"{output_dir}/production_all_bands_ratio_interactive.html",
    title_prefix="Production Ratio",
    vmin=0,
    vmax=0.8
)


Plotting comprehension timecourses...
Creating interactive HTML visualization with glass brains: Comprehension
  Rendering 41 frames...
  Rendering frame 1/41
  Rendering frame 6/41
  Rendering frame 11/41
  Rendering frame 16/41
  Rendering frame 21/41
  Rendering frame 26/41
  Rendering frame 31/41
  Rendering frame 36/41
  Rendering frame 41/41
  Creating interactive Plotly figure...
  ✓ Saved interactive HTML to: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comprehension_all_bands_interactive.html

Plotting production timecourses...
Creating interactive HTML visualization with glass brains: Production
  Rendering 41 frames...
  Rendering frame 1/41
  Rendering frame 6/41
  Rendering frame 11/41
  Rendering frame 16/41
  Rendering frame 21/41
  Rendering frame 26/41
  Rendering frame 31/41
  Rendering frame 36/41
  Rendering frame 41/41
  Creating interactive Plotly figure...
  ✓ Saved interactive HTML to: /scratch/gpfs/HASSON/ij9216/projects/code/247/24

## Generate Videos

In [9]:
anim_comp_future = pu.create_video(
    df=comp_future,
    lags=lag_columns,
    cmap=future_cmap,
    title="Comprehension - Future Context",
    output_path=f"{output_dir}/comp_future_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

anim_comp_past = pu.create_video(
    df=comp_past,
    lags=lag_columns[::-1],  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Comprehension - Past Context",
    output_path=f"{output_dir}/comp_past_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

anim_comp_word = pu.create_video(
    df=comp_word,
    lags=lag_columns,
    cmap=word_cmap,
    title="Comprehension - Word Context",
    output_path=f"{output_dir}/comp_word_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

anim_prod_future = pu.create_video(
    df=prod_future,
    lags=lag_columns,
    cmap=future_cmap,
    title="Production - Future Context",
    output_path=f"{output_dir}/prod_future_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

anim_prod_past = pu.create_video(
    df=prod_past,
    lags=lag_columns[::-1],  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Production - Past Context",
    output_path=f"{output_dir}/prod_past_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

anim_prod_word = pu.create_video(
    df=prod_word,
    lags=lag_columns,
    cmap=word_cmap,
    title="Production - Word Context",
    output_path=f"{output_dir}/prod_word_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

Creating video: Comprehension - Future Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comp_future_video.mp4
  Video saved!

Creating video: Comprehension - Past Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comp_past_video.mp4
  Video saved!

Creating video: Comprehension - Word Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comp_word_video.mp4
  Video saved!

Creating video: Production - Future Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/prod_future_video.mp4
  Video saved!

Creating video: Production - Past Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/prod_past_video.mp4
  Video saved!

Creating video: Production - Word Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/2

## Generate Ratio Videos

In [10]:
anim_prod_past_ratio = pu.create_video(
    df=prod_past_ratio,
    lags=lag_columns,  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Production - Past/Joint Ratio",
    output_path=f"{output_dir}/prod_past_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

anim_prod_future_ratio = pu.create_video(
    df=prod_future_ratio,
    lags=lag_columns,
    cmap=future_cmap,
    title="Production - Future/Joint Ratio",
    output_path=f"{output_dir}/prod_future_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

anim_prod_word_ratio = pu.create_video(
    df=prod_word_ratio,
    lags=lag_columns,
    cmap=word_cmap,
    title="Production - Word/Joint Ratio",
    output_path=f"{output_dir}/prod_word_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

anim_comp_past_ratio = pu.create_video(
    df=comp_past_ratio,
    lags=lag_columns,  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Comprehension - Past/Joint Ratio",
    output_path=f"{output_dir}/comp_past_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

anim_comp_future_ratio = pu.create_video(
    df=comp_future_ratio,
    lags=lag_columns,
    cmap=future_cmap,
    title="Comprehension - Future/Joint Ratio",
    output_path=f"{output_dir}/comp_future_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

anim_comp_word_ratio = pu.create_video(
    df=comp_word_ratio,
    lags=lag_columns,
    cmap=word_cmap,
    title="Comprehension - Word/Joint Ratio",
    output_path=f"{output_dir}/comp_word_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Production - Past/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/prod_past_ratio_video.mp4
  Video saved!

Creating video: Production - Future/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/prod_future_ratio_video.mp4
  Video saved!

Creating video: Production - Word/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/prod_word_ratio_video.mp4
  Video saved!

Creating video: Comprehension - Past/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comp_past_ratio_video.mp4
  Video saved!

Creating video: Comprehension - Future/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos/comp_future_ratio_video.mp4
  Video saved!

Creating video: Comprehension - Word/Joint Ratio
  Frames: 41

## Sandbox

In [14]:
prod_fut_sorted = prod_future_ratio.sort_values(by=-1000, ascending=False)

prod_word_sorted = prod_word_ratio.sort_values(by=-1000, ascending=False)

prod_word_sorted

,subject,electrode,roi,x,y,z,max_joint,-1000,-950,-900,...,550,600,650,700,750,800,850,900,950,1000
23,798,G88,IFG,-64.000000,22.000000,21.000000,0.190008,0.779527,0.815345,0.838391,...,0.034155,0.031945,0.032336,0.033587,0.028805,0.027029,0.028216,0.031559,0.025178,0.024586
9,798,G8,preCG,-44.400000,-3.200000,62.800000,0.270291,0.728112,0.747737,0.763226,...,0.053436,0.067030,0.065300,0.103025,0.106640,0.101345,0.096442,0.085737,0.048616,0.035152
16,798,G7,preCG,-49.333333,5.333333,58.000000,0.235750,0.702070,0.731251,0.738384,...,0.130517,0.063597,0.062354,0.058404,0.057709,0.052806,0.047576,0.043114,0.044073,0.039437
2,798,G15,preCG,-57.000000,0.000000,51.000000,0.238125,0.677582,0.704573,0.758701,...,0.166172,0.124118,0.142470,0.092815,0.095024,0.057458,0.027133,0.023355,0.018736,0.016232
81,676,EEGG_14REF,preCG,-62.000000,-2.000000,43.000000,0.219870,0.572591,0.589756,0.632643,...,0.147563,0.111174,0.103569,0.110902,0.118321,0.113890,0.067813,0.060380,0.055050,0.051866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,7170,LGB113,STG,-72.000000,-6.000000,0.000000,0.160005,-0.009790,-0.000053,0.003761,...,0.121600,0.050798,0.035957,0.017100,0.005203,-0.012206,-0.026306,-0.032491,-0.027234,-0.024736
223,7170,RAT6,None,53.000000,24.000000,-19.000000,0.107134,-0.015182,0.005798,0.035866,...,0.018288,0.019238,0.014757,0.020246,0.029383,0.041436,0.035160,0.033457,0.016300,-0.003195
184,7170,LPT1,fusiform,-46.666667,-62.000000,-23.333333,0.151800,-0.015671,-0.017180,-0.009299,...,-0.001126,0.001690,-0.001712,0.001555,0.004717,0.005896,0.001799,-0.001942,-0.002908,-0.004346
207,7170,LGB114,STG,-72.857143,-17.142857,5.142857,0.138484,-0.024694,-0.013495,0.001857,...,0.083069,0.071769,0.069577,0.070870,0.039442,0.029996,0.018817,0.014756,0.007695,0.001351


In [ ]:
# def create_combined_video(df_list, lags, cmap_list, titles, output_path, vmin=0, vmax=0.8, fps=3):
#     """
#     Create a combined video with 3 rows, each showing a different ratio video.
    
#     Parameters:
#     - df_list: List of dataframes for the 3 rows.
#     - lags: List of lag columns to iterate over.
#     - cmap_list: List of colormaps for each row.
#     - titles: List of titles for each row.
#     - output_path: Path to save the output video.
#     - vmin, vmax: Color scale limits.
#     - fps: Frames per second for the video.
#     """
#     print(f"Creating combined video: {output_path}")
#     print(f"  Frames: {len(lags)}")
    
#     # Get coordinates (same for all rows)
#     coords_list = [df[['x', 'y', 'z']].values for df in df_list]
    
#     # Set up the figure and subplots
#     fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12), constrained_layout=True)
    
#     def update_frame(frame_idx):
#         """Update function for animation."""
#         lag = lags[frame_idx]
        
#         for row, ax in enumerate(axes):
#             # Clear the axis completely to avoid clutter
#             ax.cla()
            
#             # Plot the markers for the current lag
#             values = df_list[row][lag].values
#             plotting.plot_markers(
#                 node_values=values,
#                 node_coords=coords_list[row],
#                 node_size=50,
#                 node_cmap=cmap_list[row],
#                 node_vmin=vmin,
#                 node_vmax=vmax,
#                 display_mode='lzry',
#                 colorbar=False,
#                 axes=ax
#             )
#             ax.set_title(f"{titles[row]} - Lag: {lag} ms", fontsize=12)
        
#         # Set the overall title for the figure
#         fig.suptitle("Combined Ratio Video", fontsize=16, y=0.95)
#         return fig,
    
#     # Create animation
#     anim = animation.FuncAnimation(
#         fig, 
#         update_frame, 
#         frames=len(lags),
#         interval=1000/fps,  # milliseconds per frame
#         blit=False
#     )
    
#     # Save video
#     writer = animation.FFMpegWriter(fps=fps, bitrate=1800)
#     anim.save(output_path, writer=writer)
    
#     plt.close(fig)
#     print(f"  Combined video saved at {output_path}\n")
#     return anim

# # Example usage:
# combined_video = create_combined_video(
#     df_list=[prod_past_ratio, prod_future_ratio, prod_word_ratio],
#     lags=lag_columns,
#     cmap_list=[past_cmap, future_cmap, word_cmap],
#     titles=["Production - Past/Joint Ratio", "Production - Future/Joint Ratio", "Production - Word/Joint Ratio"],
#     output_path=f"{output_dir}/combined_ratio_video.mp4",
#     vmin=0,
#     vmax=0.8,
#     fps=3
# )

Creating combined video: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4
  Frames: 41
  Combined video saved at /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4

  Combined video saved at /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4

